# One incident. Two models. A router.

**Can we solve the same incident with less model spend?**

Compare efficient-only, capable-only, and **NVIDIA NeMo Switchyard** routing.
Watch the **diagnosis, model choices, time, and cost**.


## Setup · run before presenting

Model roles and illustrative token prices are configured below. Live runs use your endpoint; enabled LangSmith tracing uploads prompts, responses, and tool data.

Set `MODE = "rehearsal"` and `ENABLE_LANGSMITH = False` for scripted practice.


In [ ]:
from getpass import getpass
from copy import deepcopy
import os
from dotenv import load_dotenv
from IPython.display import display, HTML
import importlib
import incident_demo
importlib.reload(incident_demo)  # Pick up edits in an existing notebook kernel.
from incident_demo import (
    ROOT, load_settings, load_data, run_case, run_comparison, save_results,
    show_demo_case, show_demo_comparison, demo_progress_printer,
    demo_model_names, demo_role_table, show_case, price_demo_results,
)

load_dotenv(ROOT / ".env", override=True)


In [ ]:
MODE = "rehearsal"
ALLOW_PAID_CALLS = False
ENABLE_LANGSMITH = False  # Set True to upload future runs to your LangSmith project.
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "aws-nvidia-langchain-workshop") # aws-nvidia-langchain-workshop
PROFILE = ROOT / "configs/model-profiles/demo-models.toml"
STRATEGY = ROOT / "configs/routing/task-classifier.toml"

settings = load_settings(PROFILE, STRATEGY, mode=MODE, allow_paid_calls=ALLOW_PAID_CALLS)
if MODE == "live" and ALLOW_PAID_CALLS:
    for key_env in {settings.efficient.key_env, settings.frontier.key_env, settings.judge.key_env}:
        if not os.getenv(key_env):
            os.environ[key_env] = getpass(f"{key_env}: ")
settings.enable_langsmith = ENABLE_LANGSMITH
settings.langsmith_project = LANGSMITH_PROJECT
if ENABLE_LANGSMITH and not os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LANGSMITH_API_KEY: ")
settings.validate()
models = demo_model_names(settings)
print(f"{MODE.upper()} · Tracing: {'on' if ENABLE_LANGSMITH else 'off'}")
display(demo_role_table(settings))


### Illustrative prices

Editable USD per million tokens. Costs include the judge; these rates are scenario assumptions.


In [ ]:
DEMO_PRICES = {
    settings.efficient.model: {
        "input_per_million": 0.05,
        "output_per_million": 0.20,
        "cached_input_per_million": 0.03,
        "cache_write_per_million": 0.05,
    },
    settings.frontier.model: {
        "input_per_million": 2.00,
        "output_per_million": 10.00,
        "cached_input_per_million": 0.10,
        "cache_write_per_million": 2.50,
    },
}
# USD per million tokens, including cache rates. Add a rate if the judge uses a third model.


## 1. The incident

**What evidence would you check first?**

The agent reads the ticket, logs, metrics, and troubleshooting guides, then proposes a fix.


In [ ]:
DEMO_CASE = "INC-004"
cases, _, _ = load_data()
ticket = next(case["ticket"] for case in cases if case["id"] == DEMO_CASE)
print("Incident:", DEMO_CASE)
print("Service:", ticket["service"])
print("Symptoms:", ticket["summary"])
print("Impact:", ticket["impact"])


## 2. Watch the routing

**Judge → choose a model → investigate → repeat.**

The judge estimates whether the efficient model can handle the next step. Switchyard applies the threshold; an unusable judgment falls back to the capable model.

Python checks the final diagnosis and evidence.


In [ ]:
demo_result = await run_case(DEMO_CASE, "switchyard", settings, on_event=demo_progress_printer)
show_demo_case(demo_result, settings)


## 3. Compare the outcomes

Same incident and tools. **Reports passed** means all predefined checks passed; time and cost include routing overhead.

Compare quality before savings. These synthetic results illustrate the workflow; the agent does not execute the fix.


In [ ]:
RUN_COMPARISON = True  # Set True to run the three approaches on this one incident.
REPEATS = 1  # Set 5 for a pilot; each repeat runs three tasks per incident.
COMPARISON_CASES = [DEMO_CASE]  # Add other incident IDs when testing reliability.
ORDER_SEED = 41  # Controls task order, not model generation.
REPLAY_PATH = None  # Set to a saved runs/live-*.json path for a stable offline comparison.
comparison_results = []
if REPLAY_PATH is not None:
    comparison_results = incident_demo.load_results(REPLAY_PATH)
    priced_results = comparison_results
    print("RECORDED COMPARISON ·", REPLAY_PATH.name)
    show_demo_comparison(priced_results, settings, compact=True)
    print("Recorded incidents:", sorted({r["case_id"] for r in comparison_results}))
elif RUN_COMPARISON:
    # Freeze settings and rates for every arm/repeat in this comparison.
    comparison_settings = deepcopy(settings)
    comparison_prices = deepcopy(DEMO_PRICES)
    print("Tasks:", 3 * len(COMPARISON_CASES) * REPEATS, "— fresh live calls" if MODE == "live" else "— rehearsal")
    comparison_results = await run_comparison(COMPARISON_CASES, comparison_settings,
                                              repeats=REPEATS, seed=ORDER_SEED)
    priced_results = price_demo_results(comparison_results, comparison_prices)
    show_demo_comparison(priced_results, comparison_settings, compact=True)
    saved_path = save_results(priced_results, comparison_settings, ROOT / "runs")
    print("Saved:", saved_path.name)
else:
    print("Comparison is off. Set RUN_COMPARISON = True when ready.")


## Takeaway

**Use routing when it preserves answer quality at a worthwhile cost and latency.**

<details><summary>Presenter tools</summary>

Set `REPRICE = True` below to update existing results using the current rates, or `SHOW_DETAILS = True` to inspect the investigation. Neither makes new model calls. Setup, replay instructions, and troubleshooting are in [Presenter notes](PRESENTER_NOTES.md).

</details>


In [ ]:
REPRICE = False
SHOW_DETAILS = False
if REPRICE and comparison_results:
    priced_results = price_demo_results(comparison_results, DEMO_PRICES)
    show_demo_comparison(priced_results, settings, compact=True)
if SHOW_DETAILS:
    show_case(demo_result)
